In [1]:
# python version 3.9.16
import streamlit as st
import pandas as pd
import altair as alt

/Users/rf50/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [9]:
import os
import numpy as np
import pandas as pd
import json

import asyncio
import time

from openai import OpenAI, AsyncOpenAI, OpenAIError

In [ ]:
s

In [12]:
client = OpenAI(
    api_key="IN OTHER REPOSITORY"
)

In [15]:
# response = client.responses.create(
#     model="gpt-5",
#     input="Classify Bach, J. S. as a composer."
# )
# print(response.output_text)

Baroque-era German composer (Johann Sebastian Bach).


In [16]:
SYSTEM_PROMPT = """
You are a music metadata classification system.

Your task is to classify a composer or musical category
using ONLY the original text string provided.

Do not use external research.
Do not assume additional information beyond what can
reasonably be inferred from the name or category.

Return exactly four hierarchical tags:
style, style_sub, region, region_sub.

----------------------------
STYLE TAXONOMY
----------------------------

1. Classical

Children:
- Early Music
- Baroque
- Classical
- Romantic
- Impressionist
- Modern Classical
- Contemporary Classical

Choose the most appropriate historical period/style.

2. Hymn

For entries beginning with "Hymn":

- style = Hymn
- style_sub = the original suffix/category

Examples:
"Hymn, Christmas" -> Christmas
"Hymn, Lent" -> Lent
"Hymn, Vesper" -> Vesper
"Hymn" -> General

Preserve the specificity of the original category.
Create additional subcategories when needed.

3. Traditional

For entries beginning with "Traditional":

- style = Traditional
- style_sub = the original cultural, geographical,
  or musical designation.

Examples:
"Traditional, Irish" -> Irish
"Traditional, American" -> American
"Traditional, African American Spiritual"
    -> African American Spiritual

Preserve the original designation.
Create additional subcategories when appropriate.

4. Popular

Children are restricted to:
- Film
- Musical
- Pop
- Jazz

Examples:
"Popular, Film" -> Film
"Popular, Musical" -> Musical
"Popular, Beatles" -> Pop
"Popular, Jazz" -> Jazz

If no category is specified, default to Pop.
Do not create additional Popular subcategories.

For named composers, classify their musical identity
as accurately as possible. 

----------------------------
REGION TAXONOMY
----------------------------

Parent categories:

- Europe
- North America
- South America
- Asia
- Africa
- Australia

Children should identify the composer's country,
nationality, or cultural origin as specifically as possible.

Examples:

Bach -> Europe / German
Mozart -> Europe / Austrian
Debussy -> Europe / French
Chopin -> Europe / Polish
Grieg -> Europe / Norwegian
Gershwin -> North America / American
Tchaikovsky -> Europe / Russian
Sor -> Europe / Spanish

Create additional geographical subcategories as needed.

Use historical origin/nationality, not simply
the country where the composer worked.

For Traditional entries, use the cultural/geographical
origin specified in the original name.

----------------------------
SPECIAL RULES
----------------------------

- Classify ONLY from the original text string.
- Do not invent biographical information.
- Anonymous, Various, and unknown identities should
  receive Unknown where classification is not possible.
- For ambiguous names, make the best reasonable guess.
- Do not confuse a musical category with an individual.
- Return one parent and one child for each hierarchy.
- Never return multiple tags in one field.
"""

In [17]:
OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "style": {
            "type": "string"
        },
        "style_sub": {
            "type": "string"
        },
        "region": {
            "type": "string"
        },
        "region_sub": {
            "type": "string"
        },
        "confidence": {
            "type": "string",
            "enum": ["High", "Medium", "Low"]
        }
    },
    "required": [
        "style",
        "style_sub",
        "region",
        "region_sub",
        "confidence"
    ],
    "additionalProperties": False
}

In [18]:
def classify_composer(name):

    response = client.responses.create(
        model="gpt-6.2-sol",

        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"Classify this entry: {name}"
            }
        ],

        text={
            "format": {
                "type": "json_schema",
                "name": "composer_classification",
                "strict": True,
                "schema": OUTPUT_SCHEMA
            }
        }
    )

    return json.loads(response.output_text)

In [19]:
classify_composer("Bach, J. S.")

{'style': 'Classical',
 'style_sub': 'Baroque',
 'region': 'Europe',
 'region_sub': 'German',
 'confidence': 'High'}

In [25]:
df = pd.read_csv('composers_list.csv')
df = df.rename(columns={'Composer':'composer'})
unique_names = df["composer"].dropna().unique()

results = {}

for i, name in enumerate(unique_names):

    try:
        results[name] = classify_composer(name)

    except Exception as e:
        print(f"Error processing {name}: {e}")

    # Save progress every 25 composers
    if (i + 1) % 25 == 0:

        temp = df.copy()

        for col in [
            "style",
            "style_sub",
            "region",
            "region_sub",
            "confidence"
        ]:
            temp[col] = temp["composer"].map(
                lambda x: results.get(x, {}).get(col)
            )

        temp.to_csv("composers_progress.csv", index=False)

        print(f"Processed {i + 1} composers")

    time.sleep(0.2)


# Apply results to original dataset
for col in [
    "style",
    "style_sub",
    "region",
    "region_sub",
    "confidence"
]:

    df[col] = df["composer"].map(
        lambda x: results.get(x, {}).get(col)
    )


df.to_csv("composers_tagged.csv", index=False)

print("Classification complete!")

Processed 25 composers
Processed 50 composers
Processed 75 composers
Processed 100 composers
Processed 125 composers
Processed 150 composers
Processed 175 composers
Processed 200 composers
Processed 225 composers
Processed 250 composers
Processed 275 composers
Processed 300 composers
Processed 325 composers
Processed 350 composers
Processed 375 composers
Processed 400 composers
Processed 425 composers
Processed 450 composers
Processed 475 composers
Processed 500 composers
Processed 525 composers
Processed 550 composers
Processed 575 composers
Processed 600 composers
Processed 625 composers
Processed 650 composers
Processed 675 composers


KeyboardInterrupt: 

In [27]:
df = pd.read_csv("composers_progress.csv")
completed = df["style"].notna()
print("Completed:", completed.sum())
print("Remaining:", (~completed).sum())

Completed: 675
Remaining: 842


In [28]:
for i, row in df.iterrows():
    if pd.notna(row["style"]):
        continue
    name = row["composer"]
    try:
        result = classify_composer(name)
        for col in [
            "style",
            "style_sub",
            "region",
            "region_sub",
            "confidence"
        ]:
            df.at[i, col] = result[col]

        print(f"Completed {i}: {name}")

    except Exception as e:
        print(f"Error at {i}: {name} — {e}")

    # Save after EVERY successful classification
    if pd.notna(df.at[i, "style"]):
        df.to_csv("composers_progress.csv", index=False)

Completed 675: Traditional, Swinomish
Completed 676: Traditional, Swabian (German lol)
Completed 677: Tedesco, Martini il
Completed 678: Traditional, Southern
Completed 679: Telemann, G. P.
Completed 680: Telemann, Georg P.
Completed 681: Teutsch, Jason
Completed 682: Theunissen, Kenneth
Completed 683: Thielemans, Jean
Completed 684: Traditional, South African
Completed 685: Thomas, Augusta Read
Completed 686: Thomas, Sharon
Completed 687: Traditional, Scottish?
Completed 688: Thompson, Edward 
Completed 689: Thompson, Katherine
Completed 690: Traditional, Russian 
Completed 691: Tideman-Wijers, B.
Completed 692: Tienstra, Rens
Completed 693: Traditional, Pirates (Scottish....)
Completed 694: Timmermans, F.
Completed 695: Tindley, Charles Albert
Completed 696: Tartini, G.
Completed 697: Traditional, Ukrainian (Leontovich, Mykola)
Completed 698: Tarrega, F.
Completed 699: Swaan, Egmont
Completed 700: Ukranian Folk Song
Completed 701: UChicago (Evans, Mack and Rosales, Eustasio)
Complete

In [29]:
df.to_csv("composers_tagged.csv", index=False)

In [ ]:
style
style_sub
region
region_sub

In [ ]:
Responses API
 GPT-6.2 Sol.
- Send individual composer names, but automate using Python.

In [5]:
df_composers = pd.read_csv("composers_list.csv")